In [72]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Internal-Tank-Registration\source')

import os
import time
import numpy as np
from itertools import chain
import pickle
from glob import glob
from pathlib import Path
from copy import deepcopy

from multiprocessing import Pool

from cylinder_extraction.station import Station, process_internal_station
from cylinder_alignment.station_alignment import StationAlignment
from utils.write_output import write_array2txt
from utils.math_methods import project_point_onto_plane
from cylinder_extraction.config import threshold
from cylinder_extraction.cylinder_extraction import cells_classification

from cylinder_extraction.utils import detect_objects

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## For multiple directories

In [12]:
threshold["min_length"] = 1.0
threshold["cell_size"] = 1.0

scan_field_names = [('JB-569 120ft',8)]
execuated_time = {}
for field, pack_num in scan_field_names:
    time_begin = time.time()
    try:
        print('===========================')
        print(field)
        data_dir = Path(f'D:\REGISTRATION DATA\Internal - No FR/{field}')
        filename_template = '{}.e57'
        ptc_col_names =  ["x","y","z"]

        source_files = glob(str(data_dir / '*.e57')) # + glob(str(data_dir / '*.pcd'))
        station_names = []
        stations = []
        for ind, fp in enumerate(source_files):
            if 'roof' in fp.lower():
                continue
            # station name
            station_name = Path(fp).stem
            # station id
            station_id = ind

            print(f'STATION {station_name}')
            station = Station(
                name=station_name, file_path=fp, read_points=False, station_id=station_id
            )
            stations.append(station)

        # pack_num = 8
        num_stations = len(stations)
        for i in range(0, num_stations, pack_num):
            j = min(i+pack_num, num_stations)
            pack = stations[i:j]
            with Pool(pack_num) as p:
                pack = p.starmap(
                    process_internal_station,
                    zip(
                        pack,
                        [threshold]*len(pack)
                    )                    
                )
            for station in pack:
                station.clusters = None
                # write station metadata
                out_dir = os.path.join(data_dir,station.name + '-output')
                if not os.path.exists(out_dir):
                    os.mkdir(out_dir)
                # save station without points
                station.ptc = None
                out_fp = os.path.join(out_dir, f'{station.name}.pkl')
                with open(out_fp, 'wb') as f:
                    pickle.dump(station, f)
    except Exception as e:
        print(f'Failed handling scan {field}')
        print(e)
    time_end = time.time()
    lasted = time_end - time_begin
    print(f'processing took {lasted} seconds')
    execuated_time[field] = lasted
        

JB-569 120ft
STATION Tk12002_Scan_univar12001Scan_001
STATION Tk12002_Scan_univar12001Scan_002
STATION Tk12002_Scan_univar12001Scan_003
STATION Tk12002_Scan_univar12001Scan_004
STATION Tk12002_Scan_univar12001Scan_005
STATION Tk12002_Scan_univar12001Scan_006
STATION Tk12002_Scan_univar12001Scan_007
STATION Tk12002_Scan_univar12001Scan_008
STATION Tk12002_Scan_univar12001Scan_009
STATION Tk12002_Scan_univar12001Scan_011
STATION Tk12002_Scan_univar12001Scan_012
STATION Tk12002_Scan_univar12001Scan_013
STATION Tk12002_Scan_univar12001Scan_014
STATION Tk12002_Scan_univar12001Scan_016
processing took 570.5853700637817 seconds


In [13]:
execuated_time

{'JB-569 120ft': 570.5853700637817}

# DEV

In [16]:
threshold["min_length"] = 1.0
threshold["cell_size"] = 1.0

scan_field_names = [('JB-569 120ft',8)]
execuated_time = {}
for field, pack_num in scan_field_names:
    time_begin = time.time()
    try:
        print('===========================')
        print(field)
        data_dir = Path(f'D:\REGISTRATION DATA\Internal - No FR/{field}')
        filename_template = '{}.e57'
        ptc_col_names =  ["x","y","z"]

        source_files = glob(str(data_dir / '*.e57')) # + glob(str(data_dir / '*.pcd'))
        station_names = []
        stations = []
        for ind, fp in enumerate(source_files):
            if 'roof' in fp.lower():
                continue
            # station name
            station_name = Path(fp).stem
            # station id
            station_id = ind

            print(f'STATION {station_name}')
            station = Station(
                name=station_name, file_path=fp, read_points=False, station_id=station_id
            )
            stations.append(station)
    except Exception as e:
        print(f'Failed handling scan {field}')
        print(e)
        

JB-569 120ft
STATION Tk12002_Scan_univar12001Scan_001
STATION Tk12002_Scan_univar12001Scan_002
STATION Tk12002_Scan_univar12001Scan_003
STATION Tk12002_Scan_univar12001Scan_004
STATION Tk12002_Scan_univar12001Scan_005
STATION Tk12002_Scan_univar12001Scan_006
STATION Tk12002_Scan_univar12001Scan_007
STATION Tk12002_Scan_univar12001Scan_008
STATION Tk12002_Scan_univar12001Scan_009
STATION Tk12002_Scan_univar12001Scan_011
STATION Tk12002_Scan_univar12001Scan_012
STATION Tk12002_Scan_univar12001Scan_013
STATION Tk12002_Scan_univar12001Scan_014
STATION Tk12002_Scan_univar12001Scan_016


In [18]:
s = stations[2]
s.read_points()

D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft Tk12002_Scan_univar12001Scan_003.e57
[8.95695153e-01 5.76834248e-04 4.37497165e-04 4.44668044e-01]
[  0.           0.         -44.03179453]
fields
['cartesianX', 'cartesianY', 'cartesianZ', 'intensity', 'rowIndex', 'columnIndex', 'cartesianInvalidState']
[0 0 0 ... 0 0 0]


In [19]:
from utils.cell_grid import cell_grid, cell_grid_modify

In [77]:
threshold["cell_size"] = 0.2
threshold["min_ptc"] = 1
cellgrid = cell_grid(s.ptc, threshold["cell_size"] , threshold["min_ptc"])
cellgrid.subdivision()
cellgrid = cell_grid_modify(None, None, None).cellgrid_extraction(cellgrid, cell_prop="Full")

In [79]:
cells_height = cellgrid.cal_cell_height(None, gap_threshold=threshold["point_gap"])
threshold['min_column_height'] = 5
# Cell classification based on height
mask = (cells_height[:, 0] >= threshold["min_column_height"]) & (cells_height[:, 1] >= 0.5*threshold["min_tank_height"])
tank_cells_linear_ids = np.where(mask == True)[0]
ground_cells_linear_ids = np.where(mask == False)[0]

In [81]:
dim = cellgrid.ids.max(axis=0)
count = np.zeros(cellgrid.ids.max(axis=0)+1)
for i,coord in enumerate(cellgrid.ids):
    x,y = coord
    if i in ground_cells_linear_ids:
        count[x,y] = 0
    else:
        count[x,y] = cellgrid.ptc_ids[i].size


In [40]:
import plotly.express as px

In [87]:
s

{'name': 'Tk12002_Scan_univar12001Scan_003'}

In [86]:
px.imshow(count > 500)

In [ ]:
if cell_ids is None:
            cells_height = np.full((self.ids.shape[0], 2), np.inf, dtype = np.float32)
            cell_ids = np.arange(self.ids.shape[0])
        else:
            cells_height = np.zeros_like(cell_ids)

        # Calculate the height
        for count, cell_id in enumerate(cell_ids):
            # Get ptc_ids
            cell_ptc_ids = self.ptc_ids[cell_id]
            cell_ptc_z = self.points[cell_ptc_ids, 2]
            block_length = cal_block_length(cell_ptc_z, gap_threshold)
            # Cal height
            cells_height[count] = [np.max(cell_ptc_z) - np.min(cell_ptc_z), np.max(block_length)]
        # Return
        return cells_height